# Downstream FedProx pilot mu=0.01 - CWRU + HSG18

Notebook de ejecucion del bloque downstream sobre el ckpt FL FedProx pilot
mu=0.01 (`ssl_federated_pilot_fedprox_mu0_01_patchtst_phm/ckpt_final.pt`).

## Que hace

4 corridas en Colab Pro+ A100-SXM4-80GB:

1. CWRU + linear_probing (~50 min)
2. CWRU + full_finetuning lr_backbone=1e-5 (~50 min)
3. HSG18 + linear_probing (~15 min)
4. HSG18 + full_finetuning lr_backbone=1e-5 (~15 min)

Tiempo total estimado: ~2 h. Replica EXACTAMENTE el patron del bloque
FedAvg downstream (notebook `run_fl_downstream_pilot_cwru_hsg18.ipynb`,
commit `e9fb202`) salvo el ckpt evaluado y los paths de salida, para
comparar bit-a-bit FedProx pilot vs FedAvg pilot.

## Que NO hace

- **NO lanza full FL FedProx** (requiere autorizacion + stage=full).
- **NO ejecuta entrenamientos reales sin autorizacion explicita**:
  las celdas de entrenamiento traen un aviso claro y el usuario debe
  ejecutarlas a mano una por una tras confirmar GPU=A100.
- **NO toca `min_client_presence` ni el `sampling_policy`**.
- **NO ablar mu** (eso quedaria para v0.3 si v0.2 es CONDITIONAL).
- **NO reescribe ni mueve resultados FedAvg previos**
  (paths de salida distintos: `downstream_federated_pilot_fedprox_mu0_01/`).

## Comparacion esperada (3 backbones x 2 datasets x 2 modos SSL)

Tras las 4 corridas, el summary del final del notebook compara contra:

- `from_scratch` (baseline cerrado en bloque downstream central).
- `central_linear` y `central_full_1e-5` (SSL central, ckpt 100k).
- `fed_linear` y `fed_full_1e-5` (SSL FedAvg pilot, `9b6c9fb`).
- `fedprox_linear` y `fedprox_full_1e-5` (SSL FedProx pilot, **este bloque**).

## Restricciones operativas

- **GPU A100 obligatoria** (CWRU full_finetuning con C=2 y ~140k
  ventanas necesita VRAM no trivial).
- Si el smoke pre-existente FedProx no paso, parar; el ckpt no es valido.
- Si algun dry-run abajo falla, parar y diagnosticar.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. colab_init

In [ ]:
!bash /content/drive/MyDrive/fm_fl_phmd/colab_init.sh

## 3. Pull al HEAD

Debe incluir los 4 configs FedProx downstream, el test
`test_downstream_fedprox_pilot_configs.py` y este notebook.

In [ ]:
%cd /content/fm_fl_phmd
!git pull --ff-only
!git log -3 --oneline
!git status --porcelain

## 4. Verificar GPU

**Si NO es A100, parar aqui** y vuelve con sesion A100. CWRU
full_finetuning con C=2 y ~140k ventanas es exigente; en T4/L4
puede tardar > 3 h por corrida.

In [ ]:
!nvidia-smi | head -15

## 5. pytest preflight

Tres tests minimos relevantes para esta corrida:
- `test_downstream_fedprox_pilot_configs.py` (blindaje bit-a-bit de los 4 YAMLs FedProx vs FedAvg).
- `test_downstream_adaptive_batch.py` (batch adaptativo correcto por C).
- `test_downstream_metrics.py` (per-class precision/recall/F1 + cross-check sklearn).

Si esto falla, **no continuar**.

In [ ]:
!python -m pytest \
  tests/test_downstream_fedprox_pilot_configs.py \
  tests/test_downstream_adaptive_batch.py \
  tests/test_downstream_metrics.py -q

## 6. Verificar el checkpoint FedProx pilot

Assert duro: el ckpt FedProx pilot debe existir y debe ser
exactamente el del pilot v0.2 (commit `bb38367`, `pilot_pass=true`).
Si no existe o si por error apuntara al FedAvg pilot, parar.

In [ ]:
import torch
from pathlib import Path

FEDPROX_CKPT = Path(
    "/content/drive/MyDrive/fm_fl_phmd/checkpoints/"
    "ssl_federated_pilot_fedprox_mu0_01/"
    "ssl_federated_pilot_fedprox_mu0_01_patchtst_phm/"
    "ckpt_final.pt"
)
FEDAVG_CKPT = Path(
    "/content/drive/MyDrive/fm_fl_phmd/checkpoints/"
    "ssl_federated_pilot/ssl_federated_pilot_patchtst_phm/ckpt_final.pt"
)

assert FEDPROX_CKPT.is_file(), (
    f"NO existe el ckpt FedProx pilot: {FEDPROX_CKPT}. "
    "Ejecuta antes el pilot FedProx via el notebook "
    "notebooks/pretraining/run_fl_fedprox_mu0_01_smoke_pilot.ipynb"
)
assert FEDPROX_CKPT != FEDAVG_CKPT, (
    "FEDPROX_CKPT no debe coincidir con FEDAVG_CKPT; abortar."
)

!ls -lh "{FEDPROX_CKPT}"
ck = torch.load(str(FEDPROX_CKPT), map_location='cpu', weights_only=False)
assert 'model_state_dict' in ck, 'ckpt FedProx no tiene model_state_dict'
print('keys top-level:', list(ck.keys()))
sd = ck['model_state_dict']
print(f"model_state_dict: {len(sd)} tensores, primer key: {next(iter(sd))}")
for k in ('round', 'config', 'config_hash', 'git_hash', 'algorithm', 'fedprox_mu'):
    if k in ck:
        print(f"{k}: {ck[k]}")

## 7. Preparar `_stdout/`

In [ ]:
!mkdir -p /content/drive/MyDrive/fm_fl_phmd/logs/downstream_federated_pilot_fedprox_mu0_01/cwru/_stdout
!mkdir -p /content/drive/MyDrive/fm_fl_phmd/logs/downstream_federated_pilot_fedprox_mu0_01/hsg18/_stdout

## 8. Variable comun: ckpt FedProx (string, para los `!` shell)

Define `CKPT_FP_STR` una sola vez para los 4 dry-runs y las 4
corridas reales. Asi evitamos copy-paste y reducimos riesgo de
apuntar accidentalmente al ckpt FedAvg.

In [ ]:
CKPT_FP_STR = (
    "/content/drive/MyDrive/fm_fl_phmd/checkpoints/"
    "ssl_federated_pilot_fedprox_mu0_01/"
    "ssl_federated_pilot_fedprox_mu0_01_patchtst_phm/"
    "ckpt_final.pt"
)
print('CKPT_FP_STR =', CKPT_FP_STR)

## 9. Dry-runs de los 4 configs (NO entrenan)

Para cada config: lee manifest, valida shapes, carga ckpt FedProx,
construye modelo y reporta `effective_bc`, `n_channels`, params
entrenables. Si algo no cuadra, parar antes de entrenar.

In [ ]:
for cfg, mode in [
    ('training/configs/downstream_cwru_fedprox_pilot_mu0_01_linear_probing.yaml',         'linear_probing'),
    ('training/configs/downstream_cwru_fedprox_pilot_mu0_01_full_finetuning_lr1e-5.yaml', 'full_finetuning'),
    ('training/configs/downstream_hsg18_fedprox_pilot_mu0_01_linear_probing.yaml',         'linear_probing'),
    ('training/configs/downstream_hsg18_fedprox_pilot_mu0_01_full_finetuning_lr1e-5.yaml', 'full_finetuning'),
]:
    print('\n=== dry-run', mode, cfg, '===')
    !python -m training.train_downstream_classification \
        --config {cfg} \
        --mode {mode} \
        --checkpoint "{CKPT_FP_STR}" \
        --dry-run 2>&1 | tail -20

---

# AVISO IMPORTANTE

Las **4 celdas de entrenamiento** siguientes NO se ejecutan
automaticamente. Cada una entrena ~15-50 min en A100, total ~2 h.

**Antes de ejecutar cualquiera de las 4 celdas siguientes**:

- confirmar `nvidia-smi` = A100;
- confirmar que el pytest de la celda 5 paso;
- confirmar que la celda 6 imprimio el ckpt FedProx y que difiere del FedAvg;
- confirmar que los 4 dry-runs de la celda 9 imprimieron `n_channels`
  esperados, `effective_bc <= 512`, y forward sintetico OK.

Las corridas se hacen **una a una** (no en paralelo). El nombre del
stdout va a Drive con el `run_name` exacto para trazabilidad.

## 10. Corrida 1/4 - CWRU linear_probing

Tiempo estimado en A100: ~50 min (~22 train shards CWRU).

In [ ]:
import time
t0 = time.time()
!python -m training.train_downstream_classification \
  --config training/configs/downstream_cwru_fedprox_pilot_mu0_01_linear_probing.yaml \
  --mode linear_probing \
  --checkpoint "{CKPT_FP_STR}" \
  2>&1 | tee /content/drive/MyDrive/fm_fl_phmd/logs/downstream_federated_pilot_fedprox_mu0_01/cwru/_stdout/downstream_cwru_fedprox_pilot_mu0_01_linear_probing.stdout.log
print(f"\n[total] CWRU linear_probing FedProx elapsed = {time.time() - t0:.1f} s")

## 11. Corrida 2/4 - CWRU full_finetuning lr_backbone=1e-5

Tiempo estimado en A100: ~50 min.

In [ ]:
import time
t0 = time.time()
!python -m training.train_downstream_classification \
  --config training/configs/downstream_cwru_fedprox_pilot_mu0_01_full_finetuning_lr1e-5.yaml \
  --mode full_finetuning \
  --checkpoint "{CKPT_FP_STR}" \
  2>&1 | tee /content/drive/MyDrive/fm_fl_phmd/logs/downstream_federated_pilot_fedprox_mu0_01/cwru/_stdout/downstream_cwru_fedprox_pilot_mu0_01_full_finetuning_lr1e-5.stdout.log
print(f"\n[total] CWRU full_finetuning lr1e-5 FedProx elapsed = {time.time() - t0:.1f} s")

## 12. Corrida 3/4 - HSG18 linear_probing

Tiempo estimado en A100: ~15 min (~5 train shards HSG18, C=1).

In [ ]:
import time
t0 = time.time()
!python -m training.train_downstream_classification \
  --config training/configs/downstream_hsg18_fedprox_pilot_mu0_01_linear_probing.yaml \
  --mode linear_probing \
  --checkpoint "{CKPT_FP_STR}" \
  2>&1 | tee /content/drive/MyDrive/fm_fl_phmd/logs/downstream_federated_pilot_fedprox_mu0_01/hsg18/_stdout/downstream_hsg18_fedprox_pilot_mu0_01_linear_probing.stdout.log
print(f"\n[total] HSG18 linear_probing FedProx elapsed = {time.time() - t0:.1f} s")

## 13. Corrida 4/4 - HSG18 full_finetuning lr_backbone=1e-5

Tiempo estimado en A100: ~15 min.

In [ ]:
import time
t0 = time.time()
!python -m training.train_downstream_classification \
  --config training/configs/downstream_hsg18_fedprox_pilot_mu0_01_full_finetuning_lr1e-5.yaml \
  --mode full_finetuning \
  --checkpoint "{CKPT_FP_STR}" \
  2>&1 | tee /content/drive/MyDrive/fm_fl_phmd/logs/downstream_federated_pilot_fedprox_mu0_01/hsg18/_stdout/downstream_hsg18_fedprox_pilot_mu0_01_full_finetuning_lr1e-5.stdout.log
print(f"\n[total] HSG18 full_finetuning lr1e-5 FedProx elapsed = {time.time() - t0:.1f} s")

## 14. Summary FedProx + comparacion central vs FedAvg vs FedProx

Lee los 4 `run_info.json` generados por las celdas 10-13 y construye:

1. tabla FedProx por dataset y modo (macro_f1, bal_acc, acc,
   best_epoch, amp_nonfinite_grad_steps, checkpoint_loaded);
2. tabla comparativa central / FedAvg pilot / FedProx pilot;
3. recall por clase de HSG18 full_finetuning (clave para detectar
   colapso a la clase mayoritaria como en FedAvg pilot);
4. propuesta automatica GO / CONDITIONAL / NO-GO basada en los
   deltas FedProx vs FedAvg en HSG18 y CWRU.

**La propuesta es informativa; NO autoriza full FL FedProx.**

In [ ]:
import json
from pathlib import Path

base = Path('/content/drive/MyDrive/fm_fl_phmd/logs/downstream_federated_pilot_fedprox_mu0_01')
runs = {
    ('CWRU',  'linear_probing'):      base / 'cwru'  / 'downstream_cwru_fedprox_pilot_mu0_01_linear_probing'         / 'run_info.json',
    ('CWRU',  'full_finetuning_lr1e-5'): base / 'cwru'  / 'downstream_cwru_fedprox_pilot_mu0_01_full_finetuning_lr1e-5' / 'run_info.json',
    ('HSG18', 'linear_probing'):      base / 'hsg18' / 'downstream_hsg18_fedprox_pilot_mu0_01_linear_probing'         / 'run_info.json',
    ('HSG18', 'full_finetuning_lr1e-5'): base / 'hsg18' / 'downstream_hsg18_fedprox_pilot_mu0_01_full_finetuning_lr1e-5' / 'run_info.json',
}

rows = {}
for (ds, mode), p in runs.items():
    if not p.is_file():
        print(f'WARN: no encontrado {p}; saltando')
        continue
    ri = json.loads(p.read_text(encoding='utf-8'))
    tm = ri.get('test_metrics') or {}
    rows[(ds, mode)] = {
        'macro_f1':                  tm.get('macro_f1', 0.0),
        'balanced_accuracy':         tm.get('balanced_accuracy', 0.0),
        'accuracy':                  tm.get('accuracy', 0.0),
        'best_epoch':                ri.get('best_epoch', -1),
        'elapsed_seconds':           ri.get('elapsed_seconds', 0.0),
        'amp_nonfinite_grad_steps':  ri.get('amp_nonfinite_grad_steps', -1),
        'config_hash':               ri.get('config_hash', ''),
        'checkpoint_loaded':         ri.get('checkpoint_loaded', ''),
        'per_class':                 tm.get('per_class'),
        'confusion_matrix':          tm.get('confusion_matrix'),
    }

print('=== FedProx pilot downstream (4 corridas) ===')
print(f"{'dataset':<6s} {'mode':<26s} {'macro_f1':>10s} {'bal_acc':>10s} {'acc':>8s} {'best_ep':>8s} {'elapsed':>10s} {'amp_nf':>7s} {'config_hash':>20s}")
print('-' * 120)
for (ds, mode), r in rows.items():
    print(f"{ds:<6s} {mode:<26s} {r['macro_f1']:>10.4f} {r['balanced_accuracy']:>10.4f} {r['accuracy']:>8.4f} {r['best_epoch']:>8d} {r['elapsed_seconds']:>10.1f}s {r['amp_nonfinite_grad_steps']:>7d} {r['config_hash']:>20s}")

# Verificacion: ckpt cargado debe ser el FedProx, no el FedAvg.
fp_path_str = (
    '/content/drive/MyDrive/fm_fl_phmd/checkpoints/'
    'ssl_federated_pilot_fedprox_mu0_01/'
    'ssl_federated_pilot_fedprox_mu0_01_patchtst_phm/ckpt_final.pt'
)
fa_path_str = (
    '/content/drive/MyDrive/fm_fl_phmd/checkpoints/'
    'ssl_federated_pilot/ssl_federated_pilot_patchtst_phm/ckpt_final.pt'
)
for k, r in rows.items():
    ckpt = str(r['checkpoint_loaded'])
    assert fp_path_str in ckpt, f"{k}: ckpt_loaded NO contiene path FedProx\n{ckpt!r}"
    assert fa_path_str not in ckpt, f"{k}: ckpt_loaded contiene path FedAvg (no es FedProx) {ckpt!r}"
print('\nVerificado: las 4 corridas usaron el ckpt FedProx (NO el FedAvg).')

## 15. Comparativa central vs FedAvg pilot vs FedProx pilot

Valores central + FedAvg pilot estan hardcoded desde
`results/downstream/fl_pilot_vs_central/summary.json` (commit
`e9fb202`). Si se reabriera ese summary, regenerar.

**Nota (bug fix 2026-05-26)**: la version anterior de esta celda
tenia un bug en la dict comprehension de `fedprox` (sombreado de la
variable `ds`) que duplicaba los valores HSG18 sobre la columna CWRU
de la tabla comparativa. Corregido en commit `25cdd81` (cierre del bloque downstream FedProx). Los
`run_info.json` reales versionados en
`results/downstream/fl_fedprox_pilot_vs_central/` son la fuente
canonica.

In [ ]:
# Hardcoded desde results/downstream/fl_pilot_vs_central/summary.json
# (commit e9fb202 + ablacion lr1e-4 en 2028943).
central = {
    'CWRU':  {'from_scratch': 0.3503, 'linear_probing': 0.7046, 'full_finetuning_lr1e-5': 0.8292},
    'HSG18': {'from_scratch': 0.5693, 'linear_probing': 0.9056, 'full_finetuning_lr1e-5': 0.9504},
}
fedavg = {
    'CWRU':  {'linear_probing': 0.4455601714474118, 'full_finetuning_lr1e-5': 0.6635384582258551},
    'HSG18': {'linear_probing': 0.6080023926042166, 'full_finetuning_lr1e-5': 0.5546503226449412},
}
# BUG FIX 2026-05-26: la version anterior tenia `for ds in ...` en la dict
# comprehension interna, sombreando el ds externo y haciendo que cada
# dataset terminara con los 4 valores combinados (ultimo iterado gana por
# clave). Corregido: iteracion interna solo sobre `mode`.
fedprox = {
    ds: {
        mode: rows[(ds, mode)]['macro_f1']
        for mode in ['linear_probing', 'full_finetuning_lr1e-5']
        if (ds, mode) in rows
    }
    for ds in ['CWRU', 'HSG18']
}

print('=== Tabla comparativa macro_f1 test (3 backbones x 2 datasets x 2 modos) ===')
print(f"{'dataset':<6s} {'mode':<26s} {'from_scratch':>13s} {'central':>10s} {'fedavg':>10s} {'fedprox':>10s} {'d_fp_vs_fa':>11s}")
print('-' * 110)
for ds in ['CWRU', 'HSG18']:
    for mode in ['linear_probing', 'full_finetuning_lr1e-5']:
        fs = central[ds]['from_scratch']
        ce = central[ds][mode]
        fa = fedavg[ds].get(mode)
        fp = fedprox[ds].get(mode)
        d = (fp - fa) if (fp is not None and fa is not None) else None
        print(
            f"{ds:<6s} {mode:<26s} "
            f"{fs:>13.4f} {ce:>10.4f} "
            f"{(fa if fa is not None else float('nan')):>10.4f} "
            f"{(fp if fp is not None else float('nan')):>10.4f} "
            f"{(d if d is not None else float('nan')):>+11.4f}"
        )

# Recall HSG18 full_finetuning_lr1e-5: deteccion de colapso a la mayoritaria.
print('\n=== HSG18 full_finetuning_lr1e-5: confusion matrix + per_class (FedProx) ===')
k = ('HSG18', 'full_finetuning_lr1e-5')
if k in rows:
    print(json.dumps(rows[k].get('confusion_matrix'), indent=2))
    pc = rows[k].get('per_class') or {}
    rec = pc.get('recall') or []
    print(f"\nrecall clase 0 (FedProx full): {rec[0] if len(rec) >= 1 else 'n/a'}")
    print(f"recall clase 1 (FedProx full): {rec[1] if len(rec) >= 2 else 'n/a'}")
    print('referencia FedAvg full (commit e9fb202):')
    print('  recall clase 0 = 0.2417 (colapso)')
    print('  recall clase 1 = 0.9934')

## 16. Propuesta automatica GO / CONDITIONAL / NO-GO

Aplica los criterios del `fedprox_v0_2_plan.md`:

- **GO** full FedProx si:
  - `macro_f1_HSG18_full_fp - macro_f1_HSG18_full_fa >= +0.05` (+5 pp absolutos);
  - **y** `macro_f1_CWRU_full_fp >= macro_f1_CWRU_full_fa - 0.05` (no destruye CWRU);
  - **y** recall clase 0 HSG18 FedProx full > 0.30 (sin colapso).
- **CONDITIONAL** si HSG18 mejora 1-4 pp o pierde el colapso pero sin alcanzar +5 pp.
- **NO-GO** full FedProx vanilla si HSG18 sigue colapsando (recall clase 0 < 0.30).

La propuesta es **informativa**; el GO/NO-GO final lo decide el usuario
tras leer toda la evidencia. NO autoriza automaticamente.

In [ ]:
def _propuesta_decision(rows, fedavg):
    """Aplica los criterios del fedprox_v0_2_plan.md y devuelve
    (verdict, motivo). verdict in {'GO', 'CONDITIONAL', 'NO-GO'}."""
    k_hsg_full = ('HSG18', 'full_finetuning_lr1e-5')
    k_cwru_full = ('CWRU',  'full_finetuning_lr1e-5')
    if k_hsg_full not in rows or k_cwru_full not in rows:
        return ('NO-GO', 'Faltan corridas obligatorias (HSG18 o CWRU full_finetuning).')

    hsg_fp = rows[k_hsg_full]['macro_f1']
    hsg_fa = fedavg['HSG18']['full_finetuning_lr1e-5']
    cwru_fp = rows[k_cwru_full]['macro_f1']
    cwru_fa = fedavg['CWRU']['full_finetuning_lr1e-5']

    pc = rows[k_hsg_full].get('per_class') or {}
    rec = pc.get('recall') or []
    rec0_hsg_fp = rec[0] if len(rec) >= 1 else None

    delta_hsg = hsg_fp - hsg_fa
    delta_cwru = cwru_fp - cwru_fa

    if rec0_hsg_fp is not None and rec0_hsg_fp < 0.30:
        return ('NO-GO', f'HSG18 colapsa: recall clase 0 = {rec0_hsg_fp:.4f} < 0.30 (hipotesis B reconfirmada).')

    if delta_hsg >= 0.05 and delta_cwru >= -0.05:
        return ('GO', f'HSG18 +{delta_hsg:.4f} >= +0.05 y CWRU delta {delta_cwru:+.4f} sin destruir.')

    if delta_hsg >= 0.01:
        return ('CONDITIONAL', f'HSG18 mejora +{delta_hsg:.4f}, pero no llega a +0.05. CWRU delta {delta_cwru:+.4f}.')

    return ('CONDITIONAL', f'HSG18 delta {delta_hsg:+.4f} marginal o nulo. CWRU delta {delta_cwru:+.4f}. Revisar caso por caso.')

if rows:
    verdict, motivo = _propuesta_decision(rows, fedavg)
    print('=== Propuesta automatica (informativa) ===')
    print(f"verdict: {verdict}")
    print(f"motivo : {motivo}")
    print('\nRecordatorio: esta propuesta NO autoriza full FL FedProx.')
    print('La decision final la toma el usuario tras leer toda la evidencia')
    print('y revisar results/downstream/fl_fedprox_pilot_vs_central/.')
else:
    print('Sin rows aun: ejecutar las celdas 10-13 antes de evaluar el verdict.')

## 17. Cierre

Pega el output completo de las celdas 14-16 en el chat para cerrar
el bloque downstream FedProx pilot vs FedAvg pilot.

Lo que ocurrira despues, con autorizacion explicita:

1. Versionar los 4 `run_info.json` reales bit-a-bit en
   `results/downstream/fl_fedprox_pilot_vs_central/{cwru,hsg18}/{linear_probing,full_finetuning_lr1e-5}/run_info.json`,
   mismo patron que el bloque FedAvg en
   `results/downstream/fl_pilot_vs_central/`.
2. Actualizar el `summary.json` y `README.md` de ese directorio con
   los numeros reales y la propuesta de decision.
3. Decidir, en base a los criterios documentados:
   - GO full FedProx (~6 h A100, 100 k steps);
   - CONDITIONAL (caso por caso);
   - NO-GO full FedProx vanilla (aceptar opcion B `min_client_presence`
     o opcion D reportar el limite estructural).

Hasta entonces: **full FedProx NO autorizado**.